# 53 - Python aponeurosis tracking/state gate

This notebook replaces MATLAB `Region.sup/deep` aponeuroses in the final TimTrack gate.

Inputs come from the corrected notebook 52 run:

1. image-derived TimTrack geofeatures from the matching source video (`UltraTimTrack_test.mp4`),
2. Python-geofeature one-step fascicle KLT handoff,
3. Python aponeurosis KLT affine plus MATLAB-style four-endpoint Kalman update.

The main question is whether final `FL`, `PEN`, and `ANG` still pass when the final 2-state fascicle Kalman uses Python aponeurosis lines instead of MATLAB `Region.sup/deep`.

In [1]:
from __future__ import annotations

from pathlib import Path
import json
import os
import subprocess
import sys
import time

os.environ.setdefault("MPLCONFIGDIR", "/private/tmp/matplotlib")

import cv2
import numpy as np
import pandas as pd
from IPython.display import Markdown, display
from scipy.io import loadmat

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from ultrasound_tracker.geometry import line_angles_batch, line_lengths_batch, normalize_angle
from ultrasound_tracker.matlab_compat import metric_row
from ultrasound_tracker.ultratrack_klt import UltraTrackKLTConfig
from ultrasound_tracker.ultratimtrack_aponeurosis import run_matlab_aponeurosis_state_video
from ultrasound_tracker.ultratimtrack_matlab_2state import MatlabTwoStateKalmanConfig, run_matlab_2state_kalman

pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 180)

OUT = ROOT / "results" / "notebook53_python_aponeurosis_state_gate"
OUT.mkdir(parents=True, exist_ok=True)

MATLAB_EXPORT = Path("/Users/grosbedou/Documents/GitHub/UltraTimTrack/UTT_numeric_export.mat")
MATLAB_RESULT = ROOT / "data" / "matlab" / "slow_low_2.mat"
VIDEO_PATH = ROOT / "data" / "raw" / "UltraTimTrack_test.mp4"
NB52 = ROOT / "results" / "notebook52_correct_video_fixed_emask_timtrack_gate"
PY_TIMTRACK_NPZ = NB52 / "image_derived_timtrack_geofeatures_arrays.npz"
PY_KLT_NPZ = NB52 / "python_geofeature_mask_one_step_klt_arrays.npz"
APO_STATE_NPZ = OUT / "python_aponeurosis_state_arrays.npz"
MAIN_FINAL_NPZ = OUT / "python_apo_state_2state_final_arrays.npz"
STRICT_FINAL_NPZ = OUT / "strict_python_apo_state_python_prior_2state_final_arrays.npz"
PARITY_CSV = OUT / "parity_metrics.csv"

FORCE_REBUILD_APO = os.environ.get("NB53_FORCE_REBUILD", "0") == "1"

for label, path in {
    "MATLAB numeric export": MATLAB_EXPORT,
    "MATLAB result": MATLAB_RESULT,
    "source video": VIDEO_PATH,
    "NB52 TimTrack": PY_TIMTRACK_NPZ,
    "NB52 KLT": PY_KLT_NPZ,
}.items():
    print(f"{label:24s} {'OK' if path.exists() else 'MISSING'} {path}")
    if not path.exists():
        raise FileNotFoundError(path)
print("Output folder:", OUT)
print("Force aponeurosis rebuild:", FORCE_REBUILD_APO)

MATLAB numeric export    OK /Users/grosbedou/Documents/GitHub/UltraTimTrack/UTT_numeric_export.mat
MATLAB result            OK /Users/grosbedou/PycharmProjects/NDORMS/data/matlab/slow_low_2.mat
source video             OK /Users/grosbedou/PycharmProjects/NDORMS/data/raw/UltraTimTrack_test.mp4
NB52 TimTrack            OK /Users/grosbedou/PycharmProjects/NDORMS/results/notebook52_correct_video_fixed_emask_timtrack_gate/image_derived_timtrack_geofeatures_arrays.npz
NB52 KLT                 OK /Users/grosbedou/PycharmProjects/NDORMS/results/notebook52_correct_video_fixed_emask_timtrack_gate/python_geofeature_mask_one_step_klt_arrays.npz
Output folder: /Users/grosbedou/PycharmProjects/NDORMS/results/notebook53_python_aponeurosis_state_gate
Force aponeurosis rebuild: False


In [2]:
mat_root = loadmat(MATLAB_EXPORT, simplify_cells=True)["UTT_numeric_export"]
region = mat_root["Region"]
fascicle = region["Fascicle"]
parms = mat_root["parms"]

height = int(mat_root["vidHeight"])
width = int(mat_root["vidWidth"])
mat_n = int(mat_root["NumFrames"])
mm_per_px = float(mat_root["ID"]) / height
block_size = np.asarray(mat_root["BlockSize"], dtype=int).reshape(-1)
win_size = (int(block_size[1]), int(block_size[0]))

py_tim = np.load(PY_TIMTRACK_NPZ, allow_pickle=False)
py_klt = np.load(PY_KLT_NPZ, allow_pickle=False)


def matlab_pair_series(values: object, n: int | None = None) -> np.ndarray:
    out = []
    for value in np.asarray(values, dtype=object).reshape(-1):
        arr = np.asarray(value, dtype=float).reshape(-1)
        out.append(arr[:2] if arr.size >= 2 else [np.nan, np.nan])
    result = np.asarray(out, dtype=float)
    return result if n is None else result[:n]


def matlab_fascicle_segments(x_field: str, y_field: str, n: int | None = None) -> np.ndarray:
    x = matlab_pair_series(fascicle[x_field], n=n)
    y = matlab_pair_series(fascicle[y_field], n=n)
    return np.column_stack([x[:, 1], y[:, 1], x[:, 0], y[:, 0]])


def matlab_apo_segments(prefix: str, n: int | None = None) -> np.ndarray:
    x = matlab_pair_series(region[f"{prefix}_x"], n=n)
    y = matlab_pair_series(region[f"{prefix}_y"], n=n)
    return np.column_stack([x[:, 0], y[:, 0], x[:, 1], y[:, 1]])


def normalized_angles_deg(segments: np.ndarray) -> np.ndarray:
    return np.asarray([normalize_angle(v, degrees=True) for v in line_angles_batch(segments, degrees=True)], dtype=float)


def metric(label: str, reference, estimate, unit: str, target_rmse: float | None = None) -> dict:
    row = metric_row(label, reference, estimate)
    row["unit"] = unit
    row["target_rmse"] = target_rmse
    row["passes"] = bool(target_rmse is None or (np.isfinite(row["rmse"]) and row["rmse"] <= target_rmse))
    return row


mat_sup = matlab_apo_segments("sup", n=mat_n)
mat_deep = matlab_apo_segments("deep", n=mat_n)
mat_raw_klt = matlab_fascicle_segments("fas_x_original", "fas_y_original", n=mat_n)
mat_final = {
    "FL_mm": np.asarray(region["fas_length"], dtype=float).reshape(-1)[:mat_n],
    "PEN_deg": np.asarray(region["fas_pen"], dtype=float).reshape(-1)[:mat_n],
    "ANG_deg": np.asarray(region["fas_ang"], dtype=float).reshape(-1)[:mat_n],
}

image_alpha = np.asarray(py_tim["ANG_deg"], dtype=float).reshape(-1)[:mat_n]
py_super_pos = np.asarray(py_tim["super_pos"], dtype=float)[:mat_n]
py_deep_pos = np.asarray(py_tim["deep_pos"], dtype=float)[:mat_n]
py_sup_meas = np.column_stack([np.ones(mat_n), py_super_pos[:, 0], np.full(mat_n, float(width)), py_super_pos[:, 1]])
py_deep_meas = np.column_stack([np.ones(mat_n), py_deep_pos[:, 0], np.full(mat_n, float(width)), py_deep_pos[:, 1]])

py_klt_mat_prior = np.asarray(py_klt["py_masks_matlab_prior_segments"], dtype=float)[:mat_n]
py_klt_python_prior = np.asarray(py_klt["py_masks_python_timtrack_prior_segments"], dtype=float)[:mat_n]

kalman_config = MatlabTwoStateKalmanConfig(
    q_parameter=float(mat_root["Q"]),
    x_measurement_variance=float(mat_root["X"]),
    alpha_measurement_variance=float(np.asarray(mat_root["R"], dtype=float).reshape(-1)[0]),
    n_start_frames=int(mat_root["NS"]),
    run_smoother=True,
)
apo_measurement_variance = np.asarray(mat_root["R"], dtype=float).reshape(-1)[1:] * 0.01

print({
    "frames": mat_n,
    "shape": (height, width),
    "mm_per_px": mm_per_px,
    "win_size": win_size,
    "apo_measurement_variance": apo_measurement_variance.tolist(),
    "kalman_config": kalman_config,
})

{'frames': 2666, 'shape': (562, 706), 'mm_per_px': 0.09021352313167261, 'win_size': (71, 21), 'apo_measurement_variance': [0.023358260265696967, 0.007706928939020612, 0.004799160442536146, 0.004799160442536146], 'kalman_config': MatlabTwoStateKalmanConfig(q_parameter=0.01, x_measurement_variance=100.0, alpha_measurement_variance=3.055292111054342, n_start_frames=1, run_smoother=True)}


In [3]:
def arrays_to_timtrack_entries(data) -> list[dict]:
    n = len(np.asarray(data["frame"]).reshape(-1))
    entries = []
    for idx in range(n):
        entries.append({
            "frame": int(data["frame"][idx]),
            "alpha": float(data["ANG_deg"][idx]),
            "phi": float(data["PEN_deg"][idx]),
            "faslen": float(data["FL_px"][idx]),
            "betha": float(data["super_apo_angle_deg"][idx]),
            "gamma": float(data["deep_apo_angle_deg"][idx]),
            "thickness": float(data["muscle_thickness_px"][idx]),
            "super_pos": np.asarray(data["super_pos"][idx], dtype=float),
            "deep_pos": np.asarray(data["deep_pos"][idx], dtype=float),
            "x": np.asarray(data["geofeature_x"][idx], dtype=float),
            "y": np.asarray(data["geofeature_y"][idx], dtype=float),
        })
    return entries

py_geofeatures = arrays_to_timtrack_entries(py_tim)
print("Loaded Python geofeature entries:", len(py_geofeatures))

Loaded Python geofeature entries: 2666


In [4]:
if APO_STATE_NPZ.exists() and not FORCE_REBUILD_APO:
    print("Loading cached aponeurosis state:", APO_STATE_NPZ)
    with np.load(APO_STATE_NPZ, allow_pickle=False) as data:
        apo_state = {key: data[key] for key in data.files}
else:
    print("Running Python aponeurosis KLT + endpoint Kalman state estimator...")
    start = time.time()
    apo_state = run_matlab_aponeurosis_state_video(
        VIDEO_PATH,
        py_geofeatures,
        py_sup_meas,
        py_deep_meas,
        super_cut=np.asarray(parms["apo"]["super"]["cut"], dtype=float).reshape(-1),
        deep_cut=np.asarray(parms["apo"]["deep"]["cut"], dtype=float).reshape(-1),
        q_parameter=float(mat_root["Q"]),
        measurement_variance=apo_measurement_variance,
        config=UltraTrackKLTConfig(lk_win_size=win_size),
        limit=mat_n,
        progress_every=500,
    )
    print(f"Aponeurosis state runtime: {time.time() - start:.1f}s")
    np.savez(
        APO_STATE_NPZ,
        frame=np.arange(len(apo_state["super_lines"]), dtype=np.int32),
        **{key: value for key, value in apo_state.items()},
    )
    print("Saved:", APO_STATE_NPZ)

py_sup_state = np.asarray(apo_state["super_lines"], dtype=float)[:mat_n]
py_deep_state = np.asarray(apo_state["deep_lines"], dtype=float)[:mat_n]
py_sup_prior = np.asarray(apo_state["prior_super_lines"], dtype=float)[:mat_n]
py_deep_prior = np.asarray(apo_state["prior_deep_lines"], dtype=float)[:mat_n]
print({
    "affine_rate": float(np.mean(np.asarray(apo_state["affine_ok"], dtype=bool)[1:])),
    "mean_points": float(np.mean(apo_state["points_count"][1:])),
    "mean_tracked": float(np.mean(apo_state["tracked_count"][1:])),
    "mean_inliers": float(np.mean(apo_state["inlier_count"][1:])),
})

Running Python aponeurosis KLT + endpoint Kalman state estimator...


aponeurosis state processed 501/2666


aponeurosis state processed 1001/2666


aponeurosis state processed 1501/2666


aponeurosis state processed 2001/2666


aponeurosis state processed 2501/2666


aponeurosis state processed 2666/2666
Aponeurosis state runtime: 21.6s
Saved: /Users/grosbedou/PycharmProjects/NDORMS/results/notebook53_python_aponeurosis_state_gate/python_aponeurosis_state_arrays.npz
{'affine_rate': 1.0, 'mean_points': 1015.0146341463415, 'mean_tracked': 1014.992120075047, 'mean_inliers': 1014.9763602251408}


In [5]:
def apo_metric_rows(name: str, sup_lines: np.ndarray, deep_lines: np.ndarray) -> list[dict]:
    rows = [
        metric(f"{name}_super_y1_px", mat_sup[:, 1], sup_lines[:, 1], "px", 2.0),
        metric(f"{name}_super_y2_px", mat_sup[:, 3], sup_lines[:, 3], "px", 2.0),
        metric(f"{name}_deep_y1_px", mat_deep[:, 1], deep_lines[:, 1], "px", 2.0),
        metric(f"{name}_deep_y2_px", mat_deep[:, 3], deep_lines[:, 3], "px", 2.0),
        metric(f"{name}_super_angle_deg", normalized_angles_deg(mat_sup), normalized_angles_deg(sup_lines), "deg", 1.0),
        metric(f"{name}_deep_angle_deg", normalized_angles_deg(mat_deep), normalized_angles_deg(deep_lines), "deg", 1.0),
    ]
    for row in rows:
        row["variant"] = name
    return rows

apo_rows = []
for name, sup, deep in [
    ("direct_timtrack_apo", py_sup_meas, py_deep_meas),
    ("klt_prior_apo", py_sup_prior, py_deep_prior),
    ("klt_kalman_apo", py_sup_state, py_deep_state),
]:
    apo_rows.extend(apo_metric_rows(name, sup, deep))

apo_metrics = pd.DataFrame(apo_rows)
apo_metrics_path = OUT / "python_aponeurosis_state_metrics.csv"
apo_metrics.to_csv(apo_metrics_path, index=False)
print("Saved:", apo_metrics_path)
display(apo_metrics.round(4))

Saved: /Users/grosbedou/PycharmProjects/NDORMS/results/notebook53_python_aponeurosis_state_gate/python_aponeurosis_state_metrics.csv


,comparison,n,bias,mae,rmse,corr,unit,target_rmse,passes,variant
0,direct_timtrack_apo_super_y1_px,2666,1.6167,2.2405,2.7432,0.6350,px,2.0,False,direct_timtrack_apo
1,direct_timtrack_apo_super_y2_px,2666,0.9159,1.1812,1.5376,0.8140,px,2.0,True,direct_timtrack_apo
2,direct_timtrack_apo_deep_y1_px,2666,0.4466,0.5918,0.8131,0.9914,px,2.0,True,direct_timtrack_apo
3,direct_timtrack_apo_deep_y2_px,2666,-0.2343,0.4364,0.6029,0.9978,px,2.0,True,direct_timtrack_apo
4,direct_timtrack_apo_super_angle_deg,2666,0.0569,0.2035,0.2752,0.7157,deg,1.0,True,direct_timtrack_apo
5,direct_timtrack_apo_deep_angle_deg,2666,0.0553,0.0765,0.1040,0.9774,deg,1.0,True,direct_timtrack_apo
6,klt_prior_apo_super_y1_px,2666,1.4410,1.4459,1.6316,0.9394,px,2.0,True,klt_prior_apo
7,klt_prior_apo_super_y2_px,2666,1.0138,1.0224,1.1156,0.9751,px,2.0,True,klt_prior_apo
8,klt_prior_apo_deep_y1_px,2666,0.3892,0.4485,0.5832,0.9964,px,2.0,True,klt_prior_apo
9,klt_prior_apo_deep_y2_px,2666,-0.2206,0.3214,0.4276,0.9991,px,2.0,True,klt_prior_apo


In [6]:
variants = {
    "direct_timtrack_apo_matlab_klt_prior": {
        "description": "Python direct TimTrack aponeuroses + Python mask KLT handoff with MATLAB raw KLT prior",
        "sup": py_sup_meas,
        "deep": py_deep_meas,
        "klt": py_klt_mat_prior,
    },
    "apo_klt_prior_matlab_klt_prior": {
        "description": "Python aponeurosis KLT prediction only + Python mask KLT handoff with MATLAB raw KLT prior",
        "sup": py_sup_prior,
        "deep": py_deep_prior,
        "klt": py_klt_mat_prior,
    },
    "apo_state_matlab_klt_prior": {
        "description": "Python aponeurosis KLT+Kalman state + Python mask KLT handoff with MATLAB raw KLT prior",
        "sup": py_sup_state,
        "deep": py_deep_state,
        "klt": py_klt_mat_prior,
    },
    "strict_apo_state_python_klt_prior": {
        "description": "Python aponeurosis KLT+Kalman state + Python TimTrack fascicle prior",
        "sup": py_sup_state,
        "deep": py_deep_state,
        "klt": py_klt_python_prior,
    },
}

kalman_results = {}
for name, spec in variants.items():
    kalman_results[name] = run_matlab_2state_kalman(
        np.asarray(spec["klt"], dtype=float)[:mat_n],
        image_alpha[:mat_n],
        np.asarray(spec["sup"], dtype=float)[:mat_n],
        np.asarray(spec["deep"], dtype=float)[:mat_n],
        config=kalman_config,
        mm_per_pixel=mm_per_px,
    )

rows = []
for name, result in kalman_results.items():
    for key, unit, target in [("FL_mm", "mm", 2.0), ("PEN_deg", "deg", 1.0), ("ANG_deg", "deg", 1.0)]:
        row = metric(f"{name}_{key}", mat_final[key], result[key], unit, target)
        row["variant"] = name
        row["description"] = variants[name]["description"]
        rows.append(row)

final_metrics = pd.DataFrame(rows)
final_metrics = final_metrics[["variant", "description", "comparison", "unit", "n", "bias", "mae", "rmse", "corr", "target_rmse", "passes"]]
final_metrics_path = OUT / "python_apo_replacement_final_metrics.csv"
final_metrics.to_csv(final_metrics_path, index=False)
print("Saved:", final_metrics_path)
display(final_metrics.round(4))

summary_rows = []
for name in variants:
    group = final_metrics[final_metrics["variant"] == name]
    summary_rows.append({
        "variant": name,
        "description": variants[name]["description"],
        "FL_rmse_mm": float(group.loc[group["comparison"] == f"{name}_FL_mm", "rmse"].iloc[0]),
        "PEN_rmse_deg": float(group.loc[group["comparison"] == f"{name}_PEN_deg", "rmse"].iloc[0]),
        "ANG_rmse_deg": float(group.loc[group["comparison"] == f"{name}_ANG_deg", "rmse"].iloc[0]),
        "passes_final_gate": bool(group["passes"].all()),
    })
summary_df = pd.DataFrame(summary_rows)
summary_table_path = OUT / "python_apo_replacement_variant_summary.csv"
summary_df.to_csv(summary_table_path, index=False)
print("Saved:", summary_table_path)
display(summary_df.round(4))

Saved: /Users/grosbedou/PycharmProjects/NDORMS/results/notebook53_python_aponeurosis_state_gate/python_apo_replacement_final_metrics.csv


,variant,description,comparison,unit,n,bias,mae,rmse,corr,target_rmse,passes
0,direct_timtrack_apo_matlab_klt_prior,Python direct TimTrack aponeuroses + Python ma...,direct_timtrack_apo_matlab_klt_prior_FL_mm,mm,2666,-0.8946,0.9581,1.1521,0.9971,2.0,True
1,direct_timtrack_apo_matlab_klt_prior,Python direct TimTrack aponeuroses + Python ma...,direct_timtrack_apo_matlab_klt_prior_PEN_deg,deg,2666,0.3619,0.4082,0.4778,0.9976,1.0,True
2,direct_timtrack_apo_matlab_klt_prior,Python direct TimTrack aponeuroses + Python ma...,direct_timtrack_apo_matlab_klt_prior_ANG_deg,deg,2666,0.4172,0.4407,0.5095,0.9981,1.0,True
3,apo_klt_prior_matlab_klt_prior,Python aponeurosis KLT prediction only + Pytho...,apo_klt_prior_matlab_klt_prior_FL_mm,mm,2666,-0.9281,0.9630,1.1462,0.9975,2.0,True
4,apo_klt_prior_matlab_klt_prior,Python aponeurosis KLT prediction only + Pytho...,apo_klt_prior_matlab_klt_prior_PEN_deg,deg,2666,0.3676,0.4091,0.4772,0.9977,1.0,True
5,apo_klt_prior_matlab_klt_prior,Python aponeurosis KLT prediction only + Pytho...,apo_klt_prior_matlab_klt_prior_ANG_deg,deg,2666,0.4172,0.4407,0.5095,0.9981,1.0,True
6,apo_state_matlab_klt_prior,Python aponeurosis KLT+Kalman state + Python m...,apo_state_matlab_klt_prior_FL_mm,mm,2666,-0.9280,0.9642,1.1461,0.9975,2.0,True
7,apo_state_matlab_klt_prior,Python aponeurosis KLT+Kalman state + Python m...,apo_state_matlab_klt_prior_PEN_deg,deg,2666,0.3675,0.4074,0.4751,0.9978,1.0,True
8,apo_state_matlab_klt_prior,Python aponeurosis KLT+Kalman state + Python m...,apo_state_matlab_klt_prior_ANG_deg,deg,2666,0.4172,0.4407,0.5095,0.9981,1.0,True
9,strict_apo_state_python_klt_prior,Python aponeurosis KLT+Kalman state + Python T...,strict_apo_state_python_klt_prior_FL_mm,mm,2666,-0.5998,2.3122,3.9533,0.9259,2.0,False


Saved: /Users/grosbedou/PycharmProjects/NDORMS/results/notebook53_python_aponeurosis_state_gate/python_apo_replacement_variant_summary.csv


,variant,description,FL_rmse_mm,PEN_rmse_deg,ANG_rmse_deg,passes_final_gate
0,direct_timtrack_apo_matlab_klt_prior,Python direct TimTrack aponeuroses + Python ma...,1.1521,0.4778,0.5095,True
1,apo_klt_prior_matlab_klt_prior,Python aponeurosis KLT prediction only + Pytho...,1.1462,0.4772,0.5095,True
2,apo_state_matlab_klt_prior,Python aponeurosis KLT+Kalman state + Python m...,1.1461,0.4751,0.5095,True
3,strict_apo_state_python_klt_prior,Python aponeurosis KLT+Kalman state + Python T...,3.9533,2.0571,2.0676,False


In [7]:
main_name = "apo_state_matlab_klt_prior"
strict_name = "strict_apo_state_python_klt_prior"
main = kalman_results[main_name]
strict = kalman_results[strict_name]

np.savez(
    MAIN_FINAL_NPZ,
    frame=np.arange(mat_n, dtype=np.int32),
    FL_mm=main["FL_mm"],
    PEN_deg=main["PEN_deg"],
    ANG_deg=main["ANG_deg"],
    fascicle_segments=main["fascicle_segments"],
    fascicle_end_segments=main["fascicle_end_segments"],
    X_plus=main["X_plus"],
    X_minus=main["X_minus"],
    timtrack_alpha_deg=image_alpha,
    one_step_klt_segments=py_klt_mat_prior,
    sup_apo_lines=py_sup_state,
    deep_apo_lines=py_deep_state,
)
np.savez(
    STRICT_FINAL_NPZ,
    frame=np.arange(mat_n, dtype=np.int32),
    FL_mm=strict["FL_mm"],
    PEN_deg=strict["PEN_deg"],
    ANG_deg=strict["ANG_deg"],
    fascicle_segments=strict["fascicle_segments"],
    fascicle_end_segments=strict["fascicle_end_segments"],
    X_plus=strict["X_plus"],
    X_minus=strict["X_minus"],
    timtrack_alpha_deg=image_alpha,
    one_step_klt_segments=py_klt_python_prior,
    sup_apo_lines=py_sup_state,
    deep_apo_lines=py_deep_state,
)
print("Saved:", MAIN_FINAL_NPZ)
print("Saved:", STRICT_FINAL_NPZ)

cmd = [
    sys.executable,
    str(ROOT / "scripts" / "compare_ultratimtrack_parity.py"),
    "--matlab-result", str(MATLAB_RESULT),
    "--python-utt", str(MAIN_FINAL_NPZ),
    "--python-timtrack", str(PY_TIMTRACK_NPZ),
    "--video", str(VIDEO_PATH),
    "--utt-export", str(MATLAB_EXPORT),
    "--out-csv", str(PARITY_CSV),
]
print(" ".join(cmd))
run = subprocess.run(cmd, cwd=ROOT, text=True, capture_output=True)
print(run.stdout)
if run.stderr:
    print(run.stderr)
if run.returncode != 0:
    raise RuntimeError(f"compare_ultratimtrack_parity.py failed with exit code {run.returncode}")

parity_df = pd.read_csv(PARITY_CSV)
print("Saved:", PARITY_CSV)
display(parity_df.round(4))

Saved: /Users/grosbedou/PycharmProjects/NDORMS/results/notebook53_python_aponeurosis_state_gate/python_apo_state_2state_final_arrays.npz
Saved: /Users/grosbedou/PycharmProjects/NDORMS/results/notebook53_python_aponeurosis_state_gate/strict_python_apo_state_python_prior_2state_final_arrays.npz
/Users/grosbedou/PycharmProjects/NDORMS/.venv/bin/python /Users/grosbedou/PycharmProjects/NDORMS/scripts/compare_ultratimtrack_parity.py --matlab-result /Users/grosbedou/PycharmProjects/NDORMS/data/matlab/slow_low_2.mat --python-utt /Users/grosbedou/PycharmProjects/NDORMS/results/notebook53_python_aponeurosis_state_gate/python_apo_state_2state_final_arrays.npz --python-timtrack /Users/grosbedou/PycharmProjects/NDORMS/results/notebook52_correct_video_fixed_emask_timtrack_gate/image_derived_timtrack_geofeatures_arrays.npz --video /Users/grosbedou/PycharmProjects/NDORMS/data/raw/UltraTimTrack_test.mp4 --utt-export /Users/grosbedou/Documents/GitHub/UltraTimTrack/UTT_numeric_export.mat --out-csv /Users

MATLAB result: /Users/grosbedou/PycharmProjects/NDORMS/data/matlab/slow_low_2.mat
Python final:  /Users/grosbedou/PycharmProjects/NDORMS/results/notebook53_python_aponeurosis_state_gate/python_apo_state_2state_final_arrays.npz
Python TimTrack-like: /Users/grosbedou/PycharmProjects/NDORMS/results/notebook52_correct_video_fixed_emask_timtrack_gate/image_derived_timtrack_geofeatures_arrays.npz
image_depth_mm=50.7
image_height_px=562
image_width_px=706
mm_per_pixel=0.09021352
apox_1b=[20.0, 94.0, 168.0, 242.0, 316.0, 390.0, 464.0, 538.0, 612.0, 686.0]

comparison                                n       bias        mae       rmse     corr
-------------------------------------------------------------------------------------
final_FL_mm                            2666    -0.9280     0.9642     1.1461   0.9975
final_PEN_deg                          2666     0.3675     0.4074     0.4751   0.9978
final_ANG_deg                          2666     0.4172     0.4407     0.5095   0.9981
timtrack_alpha_

,comparison,n,bias,mae,rmse,corr
0,final_FL_mm,2666,-0.9280,0.9642,1.1461,0.9975
1,final_PEN_deg,2666,0.3675,0.4074,0.4751,0.9978
2,final_ANG_deg,2666,0.4172,0.4407,0.5095,0.9981
3,timtrack_alpha_deg,2666,0.4377,0.8376,2.0101,0.9330
4,timtrack_phi_vs_python_pen_deg,2666,0.4127,0.8916,2.0228,0.9267
5,timtrack_formula_faslen_px,2666,-10.9011,19.9755,51.2138,0.9106
6,timtrack_gamma_deep_apo_deg,2666,0.0494,0.0568,0.0903,0.9838
7,timtrack_betha_super_apo_deg,2666,0.0250,0.1108,0.2303,0.8483
8,timtrack_super_pos_y1,2666,1.3560,1.6072,2.3040,0.8214
9,timtrack_super_pos_y2,2666,1.0482,1.1853,1.5025,0.8854


In [8]:
parity_targets = {
    "final_FL_mm": 2.0,
    "final_PEN_deg": 1.0,
    "final_ANG_deg": 1.0,
    "timtrack_alpha_deg": 1.0,
    "timtrack_phi_vs_python_pen_deg": 1.0,
    "timtrack_formula_faslen_px": 2.0 / mm_per_px,
    "timtrack_gamma_deep_apo_deg": 1.0,
    "timtrack_betha_super_apo_deg": 1.0,
    "timtrack_super_pos_y1": 2.0,
    "timtrack_super_pos_y2": 2.0,
    "timtrack_deep_pos_y1": 2.0,
    "timtrack_deep_pos_y2": 2.0,
}
parity_decision = parity_df.copy()
parity_decision["target_rmse"] = parity_decision["comparison"].map(parity_targets)
parity_decision["passes"] = parity_decision.apply(lambda row: bool(pd.isna(row["target_rmse"]) or row["rmse"] <= row["target_rmse"]), axis=1)
parity_decision_path = OUT / "parity_gate_decision.csv"
parity_decision.to_csv(parity_decision_path, index=False)
print("Saved:", parity_decision_path)
display(parity_decision.round(4))

final_pass = bool(parity_decision[parity_decision["comparison"].isin(["final_FL_mm", "final_PEN_deg", "final_ANG_deg"])] ["passes"].all())
apo_state_pass = bool(summary_df.loc[summary_df["variant"] == main_name, "passes_final_gate"].iloc[0])
strict_pass = bool(summary_df.loc[summary_df["variant"] == strict_name, "passes_final_gate"].iloc[0])

readiness = pd.DataFrame([
    {
        "gate": "Python aponeurosis KLT+Kalman state with validated fascicle handoff",
        "status": "final FL/PEN/ANG pass" if final_pass and apo_state_pass else "final gate fails",
        "ready_for_next": bool(final_pass and apo_state_pass),
    },
    {
        "gate": "strict Python fascicle prior + Python aponeurosis state",
        "status": "final FL/PEN/ANG pass" if strict_pass else "diagnostic variant fails",
        "ready_for_next": bool(strict_pass),
    },
])
readiness_path = OUT / "readiness.csv"
readiness.to_csv(readiness_path, index=False)
display(readiness)
print("Saved:", readiness_path)

summary = {
    "notebook": "53_python_aponeurosis_state_gate.ipynb",
    "source_video": str(VIDEO_PATH),
    "uses_matlab_aponeuroses_in_main_final_gate": False,
    "uses_matlab_raw_klt_prior_in_main_final_gate": True,
    "aponeurosis_state_npz": str(APO_STATE_NPZ),
    "main_final_npz": str(MAIN_FINAL_NPZ),
    "strict_final_npz": str(STRICT_FINAL_NPZ),
    "parity_metrics_csv": str(PARITY_CSV),
    "main_final_gate_passes": bool(final_pass and apo_state_pass),
    "strict_final_gate_passes": strict_pass,
    "main_FL_rmse_mm": float(summary_df.loc[summary_df["variant"] == main_name, "FL_rmse_mm"].iloc[0]),
    "main_PEN_rmse_deg": float(summary_df.loc[summary_df["variant"] == main_name, "PEN_rmse_deg"].iloc[0]),
    "main_ANG_rmse_deg": float(summary_df.loc[summary_df["variant"] == main_name, "ANG_rmse_deg"].iloc[0]),
    "apo_affine_rate": float(np.mean(np.asarray(apo_state["affine_ok"], dtype=bool)[1:])),
}
summary_path = OUT / "summary.json"
summary_path.write_text(json.dumps(summary, indent=2))
print("Saved:", summary_path)
print(json.dumps(summary, indent=2))

decision = "passes" if summary["main_final_gate_passes"] else "does not pass yet"
display(Markdown(f"**Decision:** replacing MATLAB aponeuroses with Python aponeurosis state {decision} for the main final gate."))

Saved: /Users/grosbedou/PycharmProjects/NDORMS/results/notebook53_python_aponeurosis_state_gate/parity_gate_decision.csv


,comparison,n,bias,mae,rmse,corr,target_rmse,passes
0,final_FL_mm,2666,-0.9280,0.9642,1.1461,0.9975,2.0000,True
1,final_PEN_deg,2666,0.3675,0.4074,0.4751,0.9978,1.0000,True
2,final_ANG_deg,2666,0.4172,0.4407,0.5095,0.9981,1.0000,True
3,timtrack_alpha_deg,2666,0.4377,0.8376,2.0101,0.9330,1.0000,False
4,timtrack_phi_vs_python_pen_deg,2666,0.4127,0.8916,2.0228,0.9267,1.0000,False
5,timtrack_formula_faslen_px,2666,-10.9011,19.9755,51.2138,0.9106,22.1696,False
6,timtrack_gamma_deep_apo_deg,2666,0.0494,0.0568,0.0903,0.9838,1.0000,True
7,timtrack_betha_super_apo_deg,2666,0.0250,0.1108,0.2303,0.8483,1.0000,True
8,timtrack_super_pos_y1,2666,1.3560,1.6072,2.3040,0.8214,2.0000,False
9,timtrack_super_pos_y2,2666,1.0482,1.1853,1.5025,0.8854,2.0000,True


,gate,status,ready_for_next
0,Python aponeurosis KLT+Kalman state with valid...,final FL/PEN/ANG pass,True
1,strict Python fascicle prior + Python aponeuro...,diagnostic variant fails,False


Saved: /Users/grosbedou/PycharmProjects/NDORMS/results/notebook53_python_aponeurosis_state_gate/readiness.csv
Saved: /Users/grosbedou/PycharmProjects/NDORMS/results/notebook53_python_aponeurosis_state_gate/summary.json
{
  "notebook": "53_python_aponeurosis_state_gate.ipynb",
  "source_video": "/Users/grosbedou/PycharmProjects/NDORMS/data/raw/UltraTimTrack_test.mp4",
  "uses_matlab_aponeuroses_in_main_final_gate": false,
  "uses_matlab_raw_klt_prior_in_main_final_gate": true,
  "aponeurosis_state_npz": "/Users/grosbedou/PycharmProjects/NDORMS/results/notebook53_python_aponeurosis_state_gate/python_aponeurosis_state_arrays.npz",
  "main_final_npz": "/Users/grosbedou/PycharmProjects/NDORMS/results/notebook53_python_aponeurosis_state_gate/python_apo_state_2state_final_arrays.npz",
  "strict_final_npz": "/Users/grosbedou/PycharmProjects/NDORMS/results/notebook53_python_aponeurosis_state_gate/strict_python_apo_state_python_prior_2state_final_arrays.npz",
  "parity_metrics_csv": "/Users/gros

**Decision:** replacing MATLAB aponeuroses with Python aponeurosis state passes for the main final gate.